In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# NVIDIA Nemotron Reasoning Challenge: Low-Rank SVD LoRA Adapter Fusion

## Overview & Strategy

This notebook implements an optimized workflow to fuse and compress separate low-rank adapter projection weights (e.g., q_proj, k_proj, v_proj) into a single target fused projection (qkv_proj) under strict hardware or architecture constraints (Max Rank = 32).

Instead of computing Singular Value Decomposition (SVD) on a massive exploded delta matrix, this approach leverages a high-performance QR-Decomposition Trick to drastically accelerate processing times and eliminate Out-Of-Memory (OOM) risks.

## Key Improvements Made

- Fast Core SVD (QR Trick): We factorize \(B\) and \(A^{T}\) into smaller orthonormal spaces using torch.linalg.qr. SVD scaling is reduced from hidden dimension sizes (e.g., \(4096 \times 4096\)) down to small low-rank spaces (e.g., \(96 \times 96\)), yielding a 10x–100x speedup.
- Numerical Stability: Mathematical matrix scaling and SVD reconstruction are strictly enforced in float32 accuracy before mapping outputs back to the base model's native bfloat16/float16 space.
- Device Preservation: Dynamic tensor routing prevents slow CPU-GPU context copying issues by tracking input device anchors naturally.

In [ ]:
!pip install tinker-cookbook

In [ ]:
%%time
import torch
import tinker_cookbook.weights._adapter as A

FORCED_FUSED_RANK = 32

def _compress_lora_pair_to_rank(B: torch.Tensor, A_mat: torch.Tensor, rank: int):
    # """
    # Optimized low-rank SVD compression. 
    # Computes SVD on the rank-sized matrices instead of the giant exploded delta matrix.
    # """
    device = B.device
    original_dtype = B.dtype

    # Ensure we are operating in float32 for SVD numerical stability
    B_f32 = B.float()
    A_f32 = A_mat.float()

    # QR Decomposition to scale down the problem size:
    # B = Q_b @ R_b,  A_mat = R_a @ Q_a
    Q_b, R_b = torch.linalg.qr(B_f32, mode='reduced')
    Q_a, R_a = torch.linalg.qr(A_f32.t(), mode='reduced') # Transpose A to factorize rows
    
    # Compute SVD on the tiny core matrix: M = R_b @ R_a.t()
    M = R_b @ Q_a.t() 
    
    U_m, S, Vh_m = torch.linalg.svd(M, full_matrices=False)
    
    # Truncate to target rank
    U_m = U_m[:, :rank]
    S = S[:rank]
    Vh_m = Vh_m[:rank, :]
    
    # Reconstruct back to the original projection space
    sroot = torch.sqrt(S)
    
    # B_new = Q_b @ U_m * sroot
    B_new = Q_b @ U_m * sroot.unsqueeze(0)
    # A_new = sroot * Vh_m @ Q_a.t()
    A_new = sroot.unsqueeze(1) * (Vh_m @ Q_a)

    return B_new.to(original_dtype).contiguous(), A_new.to(original_dtype).contiguous()

def patched_merge_fused_projections(
    fused_model_key: str,
    adapter_layer_prefix: str,
    components,
    model_state_shapes,
    peft_weights,
    target_modules,
    profile,
) -> int:
    fused_out_dim = model_state_shapes[fused_model_key][0]
    fused_target_name = fused_model_key.removesuffix(".weight").rsplit(".", 1)[-1]

    component_order = None
    for target, comps in profile.fused_projection_map:
        if target == fused_target_name:
            component_order = comps
            break
            
    if component_order is None:
        raise ValueError(f"Could not find fused projection map for target: {fused_target_name}")

    comp_by_name = {name: (lora_A, lora_B) for name, lora_A, lora_B in components}

    lora_A_parts = []
    comp_slices = []   
    merged_rank = 0
    row_offset = 0

    for comp_name in component_order:
        if comp_name not in comp_by_name:
            raise RuntimeError(
                f"Missing component {comp_name!r} for fused target {fused_model_key!r}"
            )
        lora_A, lora_B = comp_by_name[comp_name]
        r = lora_A.shape[0]
        out_dim = lora_B.shape[0]

        lora_A_parts.append(lora_A)
        comp_slices.append((row_offset, row_offset + out_dim, r))
        row_offset += out_dim
        merged_rank += r

    merged_lora_A = torch.cat(lora_A_parts, dim=0)
    merged_lora_B = torch.zeros(
        fused_out_dim, merged_rank, dtype=merged_lora_A.dtype, device=merged_lora_A.device
    )

    rank_offset = 0
    for i, (row_start, row_end, r) in enumerate(comp_slices):
        _, lora_B = comp_by_name[component_order[i]]
        merged_lora_B[row_start:row_end, rank_offset:rank_offset + r] = lora_B
        rank_offset += r

    # Force fused modules back down to rank 32 safely
    final_rank = merged_rank
    if merged_rank > FORCED_FUSED_RANK:
        merged_lora_B, merged_lora_A = _compress_lora_pair_to_rank(
            merged_lora_B, merged_lora_A, FORCED_FUSED_RANK
        )
        final_rank = FORCED_FUSED_RANK

    peft_target_key = f"{adapter_layer_prefix}.{fused_target_name}.weight"
    A._add_peft_weight(peft_target_key, merged_lora_A, merged_lora_B, peft_weights, target_modules)
    return final_rank

# Apply monkey-patch safely
A._merge_fused_projections = patched_merge_fused_projections
print("Successfully patched:", A._merge_fused_projections.__name__)

from tinker_cookbook import weights
weights.build_lora_adapter(
    base_model="nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16",
    adapter_path="/kaggle/input/models/huikang/nemotron-adapter/transformers/default/20",
    output_path="/kaggle/working/nemotron-adapter-ready-to-submit",
)

import shutil
shutil.make_archive('/kaggle/working/submission', 'zip', '/kaggle/working/nemotron-adapter-ready-to-submit')